# Data extraction from ChEMBL

## Aim
Download all molecules that have been tested against the **epidermal growth factor receptor (EGFR) kinase**.
- Find ligands which were tested on a certain target
- Filter by available bioactivity data
- Calculate pIC50 values
- Merge dataframes
- Draw and display molecules

## For more details
https://github.com/volkamerlab/teachopencadd/blob/master/teachopencadd/talktorials/T001_query_chembl/talktorial.ipynb

## Instructions
Replace XXX with the appropriate code

## Configuration

In [ ]:
# Imports
# 1. Standard library imports
import math
from pathlib import Path
import sys
sys.path.append('../my_modules') # to tell where to find local modules

# 2. Third-party library imports
from chembl_webresource_client.new_client import new_client
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# import rdkit
from rdkit import Chem
from rdkit.Chem import (
    Draw,
    PandasTools
)
PandasTools.RenderImagesInAllDataFrames(images=True) # to molecules as images in DataFrames
from rdkit.Chem.Draw import IPythonConsole # needed to show molecules
# from rdkit.Chem.Draw.MolDrawing import MolDrawing, DrawingOptions # only needed if modifying defaults

# 3. Local application imports
import kernel_infos

In [ ]:
# Information about the kernel
kernel_infos.show_kernel_info()

In [ ]:
# Global variables
HERE = Path().resolve()
print(f'{HERE}')
ROOT = HERE.parent
print(f'{ROOT}')
DATA = ROOT / 'data'
print(f'{DATA}')

## Create resource objects for API access.

In [ ]:
targets_api = new_client.target
compounds_api = new_client.molecule
bioactivities_api = new_client.activity

## Target data
* Get uniprot-id (http://www.uniprot.org/uniprot/P00533) of the target of interest (EGFR kinase) from UniProt website (https://www.uniprot.org/)
* Use uniprot-id to get target information

In [ ]:
# Target
uniprot_id = 'P00533'

In [ ]:
# Get target information from ChEMBL but restricted to specified values only
targets_P00533 = targets_api.get(target_components__accession=XXX) \
                       .only('target_chembl_id', 'organism', 'pref_name', 'target_type')

In [ ]:
# The resulting object
print(type(XXX))

In [ ]:
# Get information about the resulting object
XXX?

What does a chembl_webresource_client.query_set.QuerySet consist of ?

In [ ]:
# How many elements in the resulting object ?
print(f'{XXX(targets_P00533)} target(s) found.')

In [ ]:
# Download target data from ChEMBL in a dataframe
targets_P00533_df = pd.XXX.from_records(targets_P00533)

In [ ]:
# Display the dataframe
XXX

## Select target : target_chembl_id


**CHEMBL203**: a single protein, represents the human Epidermal growth factor receptor (EGFR, also named erbB1)

In [ ]:
# Select the first target and display it
target = targets_P00533_df.iloc[XXX]
target

In [ ]:
# Save the selected CHEMBL ID in chembl_id variable and print it
XXX = target['target_chembl_id']
print(f"The target ChEMBL ID is XXX")

## Get bioactivity data

### Fetch filtered bioactivity data for the target from ChEMBL

- human proteins,
- bioactivity type IC50,
- exact measurements (relation '='), and
- binding data (assay type 'B').

In [ ]:
bioactivities_qs = bioactivities_api.filter(target_chembl_id=XXX) \
                      .filter(type='IC50') \
                      .filter(relation='=') \
                      .filter(assay_type='B') \
                      .only('activity_id', 
                      'assay_chembl_id', 
                      'assay_description', 
                      'assay_type',
                      'molecule_chembl_id', 
                      'type', 
                      'units', 
                      'relation', 
                      'value',
                      'target_chembl_id', 
                      'target_organism')

In [ ]:
print(f"type(bioactivities_qs) = {type(bioactivities_qs)}")
print(f"Number of bioactivites = {len(bioactivities_qs)}")
print(f"type(bioactivities_qs[0]) = {type(bioactivities_qs[0])}")
print(f"Number of bioactivites in bioactivities_qs[0] = {len(bioactivities_qs[0])}")

In [ ]:
## Download bioactivity data in a dataframe (could take a long time so you can use pre-downloaded data below)
# bioactivities_df = pd.DataFrame.from_dict(bioactivities_qs)

In [ ]:
# Read data from the downloaded csv file
bioactivities_ChEMBL36_zip_file_path = DATA / "EGFR_bioactivities_CHEMBL36.csv.zip"
bioactivities_df = pd.XXX(bioactivities_ChEMBL36_zip_file_path, index_col=0)

### Explore data

In [ ]:
# Print the dimensions of the dataframe and display the first 3 rows
print(f"DataFrame shape: {bioactivities_df.shape}")
bioactivities_df.head(3)

In [ ]:
# Print the data types of each column
bioactivities_df.XXX

### Preprocess data

In [ ]:
# What is the shape of bioactivities_df ?
bioactivities_df.XXX

In [ ]:
# Are there any null values ?
bioactivities_df.XXX

In [ ]:
# Delete entries with missing values (na)
bioactivities_df.XXX(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
# Are there molecule_chembl_id duplicates ?
bioactivities_df['molecule_chembl_id'].XXX

In [ ]:
# Delete duplicate entries regarding molecule_chembl_id (1 molecule tested several times)
bioactivities_df.XXX("molecule_chembl_id", keep="first", inplace=True)
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
# Convert 'value' column to numeric (forcing errors to NaN)
bioactivities_df['value'] = pd.to_numeric(XXX, errors='coerce') 

In [ ]:
# Check types after conversion 
bioactivities_df.XXX

In [ ]:
# Keep only entries with 'molar' activities
# Let's see the different unique units
bioactivities_df['units'].XXX

In [ ]:
# Select 'M' units
bioactivities_df = bioactivities_df.drop(bioactivities_df.index[~bioactivities_df['units'].str.contains('M|mol/L', case='True', regex=True)])

In [ ]:
# Check selection
print(f"{bioactivities_df.units.unique()}")
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
# Define a fonction converting the 'value' column to nM depending on the 'units' column
def convert_to_nM(DF):
    match_units = {
        'uM': 1e3,
        'nM': 1,
        'M': 1e9,
        "10'-1microM":1e2,
        "10'1 uM":1e4,
        "10'2 uM":1e5,
        '/uM':1e3,
        'mM':1e6,
        'umol/L':1e3,
        'nmol/L':1,
        '10^3 uM':1e6
    }
    
    def convert_row(row):
        unit = row['units']
        value = row['value']
        if unit in match_units:
            return value * match_units[unit]
        else:
            return None  # or handle unknown units as needed

    DF['value'] = DF.apply(convert_row, axis=1)
    DF['units'] = 'nM'
    return DF

In [ ]:
# Apply convert_to_nM function to bioactivities_df
bioactivities_df = XXX

In [ ]:
# Check the 'units' values and display the first 3 row of the dataframe 
print(f"{bioactivities_df['units'].XXX()}")
bioactivities_df.XXX

In [ ]:
# Delete entries with missing values (na)
bioactivities_df.dropna(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
# Reset index in order to keep continous values
bioactivities_df.XXX(drop=True, inplace=True)
bioactivities_df.head(3)

In [ ]:
# Rename 'value' column to 'IC50'
bioactivities_df.rename(columns={XXX: XXX}, inplace=True)
bioactivities_df.head(3)

In [ ]:
# How many activites ?
print(f"Here we have a set of {bioactivities_df.XXX} molecules IDs with respective IC_50 values for the targeted kinase.")

## Get structural data

### Get a list of ChEMBL IDs from bioactivities_df

In [ ]:
# Convert the molecule_chembl_id column into a list
molecule_chembl_ids = bioactivities_df['molecule_chembl_id'].XXX

### Fetch compounds from ChEMBL

In [ ]:
compounds_qs = compounds_api.filter(molecule_chembl_id__in = molecule_chembl_ids) \
                       .only('molecule_chembl_id', 'molecule_structures')

In [ ]:
print(f"type(compounds_qs) = {type(compounds_qs)}")
print(f"Number of compounds = {len(compounds_qs)}")
print(f"type(compounds_qs[0]) = {type(compounds_qs[0])}")

In [ ]:
# Display the keys of the first element of compounds_qs
compounds_qs[XXX].XXX

In [ ]:
# Display the first element of compounds_qs
compounds_qs[XXX]

In [ ]:
# Use predownloaded data 
compounds_ChEMBL36_parquet_file_path = DATA / "EGFR_compounds_CHEMBL36.parquet"
compounds_df = pd.read_parquet(XXX, engine='fastparquet')

In [ ]:
# Explore data: dataframe sizes and display the first 3 rows
print(f"DataFrame shape: {compounds_df.shape}")
compounds_df.head(3)

In [ ]:
# What are the types ?
compounds_df.dtypes

In [ ]:
# Display the molecule_structures of first compound
compounds_df.XXXs

### Preprocess and filter compound data

In [ ]:
# Remove entries with missing values regarding molecule_structures 
compounds_df.XXX(subset=[XXX], axis=0, how="any", inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")

In [ ]:
# Delete duplicate entries regarding molecule_chembl_id
compounds_df.XXX(XXX, keep="first", inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")
compounds_df.head(2)

In [ ]:
# Create a new column 'smiles' containing the canonical_smiles of molecule structures 
compounds_df['smiles'] = compounds_df['molecule_structures'].apply(lambda x: x['canonical_smiles'] if isinstance(x, dict) and 'canonical_smiles' in x else None)

In [ ]:
# Remove the 'molecule_structures' column
compounds_df.XXX(columns=[XXX], inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")
compounds_df.head(2)

In [ ]:
# Remove entries with missing values
compounds_df.XXX(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")

In [ ]:
# # Select canonical_smiles as molecule structure : method with iterrows
# canonical_smiles = []

# for index, row in compounds_df.iterrows():
#     mol_struct = row['molecule_structures']
#     if isinstance(mol_struct, dict) and 'canonical_smiles' in mol_struct:
#         canonical_smiles.append(mol_struct['canonical_smiles'])
#     else:
#         canonical_smiles.append(None)

# compounds_df['smiles'] = canonical_smiles
# compounds_df.drop(columns=['molecule_structures'], inplace=True)
# print(f"DataFrame shape: {compounds_df.shape}")
# compounds_df.head(2)

## Merge data
### Summary

In [ ]:
print(f"Bioactivities DataFrame shape: {bioactivities_df.shape}")
bioactivities_df.head(2)

In [ ]:
print(f"Compounds DataFrame shape: {compounds_df.shape}")
compounds_df.head(2)

### Merge both datasets

Keep : 
- ChEMBL_ID: molecule_chembl_id
- SMILES: smiles
- units: units
- IC50: IC50

In [ ]:
# Merge both dataframe in output_df on 'molecule_chembl_id' 
output_df = pd.merge(
    bioactivities_df[['molecule_chembl_id', 'IC50', 'units']], 
    compounds_df, 
    on=XXX
)

In [ ]:
# Explore data: dataframe sizes and display the first 3 rows
print(f"Output DataFrame shape: {output_df.shape}")
output_df.head(3)

In [ ]:
# Verify types
output_df.XXX

## $IC_{50}$ study

In [ ]:
# Scatterplot IC50 values versus index
plt.figure(figsize=(8, 5))
plt.XXX(output_df.XXX, output_df[XXX], color='blue', alpha=0.7)
plt.title(r"$IC_{50}$ (nM)")
plt.xlabel("Index")
plt.ylabel(r"$IC_{50}$ (nM)")
plt.show()

In [ ]:
# Scatterplot avec seaborn
plt.figure(figsize=(8, 5))
sns.XXX(
    data=XXX, 
    x=output_df.index, 
    y='IC50')
plt.title(r"$IC_{50}$ (nM)")
plt.xlabel("Index")
plt.ylabel(r"$IC_{50}$ (nM)")
plt.show()

In [ ]:
# Scatterplot with a log scale on y axis
plt.figure(figsize=(8, 5))
plt.scatter(
    output_df.index, 
    output_df['IC50'], 
    color='blue', 
    alpha=0.7
)
plt.title(r"$IC_{50}$ (nM)")
plt.xlabel("Index")
plt.ylabel(r"$IC_{50}$ (nM)")
yscale=XXX
plt.yscale(yscale)
plt.show()

### Convert $IC_{50}$ to $pIC_{50} = -\log_{10}(IC_{50}$ in M)

In [ ]:
# Create a fonction to convert IC50 (nM) in pIC50
def IC50_to_pIC50(IC50):
    """Convert IC50 values in nM to pIC50 values."""
    if IC50 > 0:
        return 9 - math.log10(IC50)
    else:
        return float('nan')

In [ ]:
# Apply the conversion to the dataframe
output_df['pIC50'] = output_df['IC50'].XXX(IC50_to_pIC50)

In [ ]:
# Explore data: dataframe sizes and display the first 3 rows
print(f"Output DataFrame shape: {output_df.shape}")
output_df.XXX

In [ ]:
# Numpy version to improve performance with big dataframes
output_df['pIC50np'] = np.where(
    output_df['IC50'] > 0,
    -np.log10(output_df['IC50']*1e-9), # 1e-9 = 10**-9
    np.nan
)

In [ ]:
print(f"Output DataFrame shape: {output_df.shape}")
output_df.head(3)

In [ ]:
# Delete pIC50np
del output_df[XXX]

# With drop
# output_df = output_df.drop(columns=['pIC50np'], inplace=True)

# With pop
# output_df.pop('pIC50np')

### $ pIC_{50} $ distribution


In [ ]:
# Matplotlib : plot an histogramm of pIC50
plt.figure(figsize=(8, 5))
plt.hist(XXX, bins=30, color='blue', alpha=0.7)
plt.title(r"$pIC_{50}$ Distribution")
plt.xlabel(r"$pIC_{50}$")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Seaborn
plt.figure(figsize=(8, 5))
sns.histplot(
    data=XXX, 
    x=XXX, 
    bins=30, 
    kde=False, 
    color='blue'
)
plt.title(r"$pIC_{50}$ Distribution")
plt.xlabel(r"$pIC_{50}$")
plt.ylabel("Frequency")
plt.show()

In [ ]:
### Boxplot with seaborn
plt.figure(figsize=(6, 8))
sns.XXX(
    data=output_df, 
    y='pIC50', 
    color='lightblue'
)
plt.title(r"$pIC_{50}$ Boxplot")
plt.ylabel(r"$pIC_{50}$")
plt.show()

## Visualise the most active compounds

### Sort compounds by pIC50 values and reset index

In [ ]:
output_df.XXX(by='pIC50', ascending=False, inplace=True)
output_df.XXX(drop=True, inplace=True)

In [ ]:
# Display the first 3 rows 
output_df.XXX

### Add a structure column and look at the molecules with the pIC50 corresponding to the most active compounds.

#### PandasTools

In [ ]:
help(PandasTools.AddMoleculeColumnToFrame)

In [ ]:
PandasTools.AddMoleculeColumnToFrame?

In [ ]:
# Add molecule column to output_df
PandasTools.XXX(
    output_df,
    smilesCol = 'smiles',
    molCol='ROMol',
)

In [ ]:
print(f"DataFrame shape: {output_df.shape}")
output_df.head(2)

#### MolFromSmiles

In [ ]:
# Add a molecule column 'ROMol2' from the 'smiles' column with a lambda function
output_df[XXX] = output_df[XXX].apply(lambda x: XXX if x else None)

In [ ]:
output_df.head(2)

In [ ]:
# Delete ROMol2
XXX output_df['ROMol2']

### Visualisation

In [ ]:
top_n = 3
output_df.head(top_n)

In [ ]:
# Draw an image of the top10 compounds, with chembl_id and pIC50 as legends
top_n = XXX

mols = [XXX for mol in output_df['ROMol'].head(top_n)]
legend = [
    (chembl_id, f"{pIC50:.2f}") 
    for chembl_id, pIC50 in XXX(
        output_df['molecule_chembl_id'].head(top_n), 
        output_df['pIC50'].head(top_n)
    )
]


img = Draw.XXX(
    mols=mols,
    molsPerRow=4,  # Number of molecules per row
    subImgSize=(300, 300),  # Size of each sub-image
    legends=[f"{chembl_id}, {pIC50}" for chembl_id, pIC50 in legend],  # Add IDs and pIC50 as legends
    useSVG=False  # Generate a PNG image
)

img

In [ ]:
# Save the image in PNG format
image_file_path = DATA / f'img_top{top_n}_EGFR_CHEMBL36.png'
with XXX(image_file_path, 'wb') as f:
    f.XXX(img.data)

## Save data without the last columns

In [ ]:
print(f"DataFrame shape: {output_df.shape}")
output_df.head(2)

In [ ]:
# Enumerate dataframe's columns
output_df.XXX

In [ ]:
# Select all the columns except the last
selected_columns = output_df.columns.to_list()[XXX]
print(f'Selected columns: {selected_columns}')

In [ ]:
# Save data in a csv file
output_csv_file_path = DATA / 'EGFR_CHEMBL36_output.csv'
output_df[selected_columns].XXX(output_csv_file_path)